# Text and spreadsheet export

Pause the web app while running GPU notebooks. These cells use the same modules as FastAPI.


Direct text extraction needs no model. XLSX uses native cells/formulas plus rendered visual analysis. CSV retains rows and quoted cell values. This adapter is new; the shared visual stages come from L6.


In [ ]:
from pathlib import Path
import sys
# Find the standalone project whether Jupyter starts at its root or notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "app/config.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from app.config import Settings
from app.processors import Processor
from app.files import atomic_text, save_json
settings = Settings(root=ROOT)
settings.initialize()
processor = Processor(settings, progress=print)



Choose a text or spreadsheet input. If a folder is empty, add your own file first; the original dataset has no XLSX files. This notebook does not invent owner data.


In [ ]:
category = "text"  # Change to "excel" for CSV/XLSX.
from app.config import FORMATS
candidates = sorted(p for p in (ROOT / "input" / category).rglob("*") if p.suffix.lower() in FORMATS[category])
print(*candidates, sep="\n")



In [ ]:
if not candidates:
    raise FileNotFoundError(f"Add a file under input/{category}/ first.")
source = candidates[0]
result = processor.process(source, category)



In [ ]:
# Write one text output with its original extension and relative path preserved.
relative = source.relative_to(ROOT / "input")
output = ROOT / "output" / (relative.as_posix() + ".txt")
atomic_text(output, result.text(relative.as_posix()))
save_json(ROOT / "runtime" / "notebook-result.json", {
    "source": str(relative), "warnings": result.warnings,
    "devices": result.devices, "details": result.details,
})
print(output)
print(output.read_text(encoding="utf-8")[:5000])

